# QMUL-SurvFace-v1 공식 데이터 준비

공식 MAT/폴더를 검증하고 gallery, mated probe, unmated probe 매니페스트를 만듭니다. MAT의 행 순서와 `protocol_index`를 바꾸지 않으며, 공식 프로토콜에 없는 `known_unknown`을 생성하지 않습니다.

| 모드 | 예상 시간 | 저장 |
| --- | ---: | --- |
| `WRITE_OUTPUTS=False` | 약 20초~1분 | 없음(전체 검증만 수행) |
| `WRITE_OUTPUTS=True` | 약 20초~2분 | `data/interim/survface/`에 원자적으로 저장 |

> **진행/체크포인트/재시작**: 역할별 검사 시작·완료와 경과 시간을 출력합니다. 검증 중 중단되면 Kernel Restart 후 처음부터 실행합니다. 저장 중 중단되면 출력 파일을 확인하고, 의도적으로 다시 만들 때만 `OVERWRITE=True`로 바꿔 처음부터 실행합니다. `training_set`은 공식 test와 섞지 않습니다.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from time import perf_counter

import numpy as np
import pandas as pd
import scipy
from IPython.display import display

from research.datasets import build_survface_official_manifest, write_survface_official_bundle

print(f"project_root: {PROJECT_ROOT}")
print(f"python: {sys.executable}")
print(f"pandas: {pd.__version__}, scipy: {scipy.__version__}")


## 1. 경로와 저장 모드

원본 루트에는 `Face_Identification_Evaluation`과 `Face_Identification_Test_Set`이 모두 있어야 합니다. 기본값은 검증 전용입니다.


In [ ]:
WRITE_OUTPUTS = False
OVERWRITE = False

SURVFACE_ROOT = PROJECT_ROOT / "data" / "raw" / "QMUL-SurvFace"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "survface"

display(pd.Series({
    "WRITE_OUTPUTS": WRITE_OUTPUTS,
    "OVERWRITE": OVERWRITE,
    "SURVFACE_ROOT": str(SURVFACE_ROOT),
    "OUTPUT_DIR": str(OUTPUT_DIR),
}, name="value").to_frame())


## 2. 공식 MAT·파일 집합·순서 검증

gallery와 mated probe는 MAT filename/ID/order를 그대로 사용합니다. unmated probe의 true identity는 공개되지 않으므로 파일별 불투명 synthetic ID만 부여합니다.


In [ ]:
started = perf_counter()
print("[RUNNING] 공식 MAT와 이미지 파일 집합 검증 시작")
bundle = build_survface_official_manifest(SURVFACE_ROOT, PROJECT_ROOT)

for role, frame in (
    ("gallery", bundle.gallery),
    ("registered_probe", bundle.registered_probes),
    ("unknown_unknown_probe", bundle.unknown_unknown_probes),
):
    actual = frame["protocol_index"].to_numpy(dtype=np.int64)
    expected = np.arange(len(frame), dtype=np.int64)
    if not np.array_equal(actual, expected):
        raise ValueError(f"{role} protocol_index가 공식 순서를 보존하지 않습니다.")
    print(f"[OK] {role}: rows={len(frame):,}, elapsed={perf_counter() - started:.1f}s")

if int(bundle.summary["known_unknown_identity_count"]) != 0:
    raise ValueError("SurvFace 공식 프로토콜의 known_unknown은 반드시 0이어야 합니다.")

display(pd.Series(bundle.summary, name="value").to_frame())
display(bundle.manifest.groupby(["protocol_role", "probe_type"], sort=False).agg(
    images=("image_id", "size"), identities=("identity_id", "nunique")
))
print(f"[COMPLETED] 전체 검증 통과, elapsed={perf_counter() - started:.1f}s")


## 3. 역할별 파일 저장

`official_manifest.csv`는 전체 역할을 포함합니다. 역할별 CSV도 같은 `protocol_index` 순서를 유지합니다.


In [ ]:
output_names = (
    "official_manifest.csv",
    "gallery.csv",
    "registered_probes.csv",
    "unknown_unknown_probes.csv",
    "gallery_identities.txt",
    "unknown_unknown_identities.txt",
    "summary.json",
)

if WRITE_OUTPUTS:
    written_paths = write_survface_official_bundle(bundle, OUTPUT_DIR, overwrite=OVERWRITE)
    print("[COMPLETED] 공식 bundle 저장 완료")
else:
    written_paths = {name: OUTPUT_DIR / name for name in output_names}
    print("[REVIEW] WRITE_OUTPUTS=False: 검증만 완료했으며 파일을 저장하지 않았습니다.")

display(pd.DataFrame([
    {"file": name, "path": str(path), "exists": path.is_file()}
    for name, path in written_paths.items()
]))


## 다음 단계

저장된 `summary.json`에서 known unknown이 0인지 다시 확인한 뒤 `00_official_protocol_and_run_freeze.ipynb`를 실행합니다. gallery는 3,000 identity의 모든 공식 gallery 이미지를 평균하는 `official_all` 정책을 사용해야 합니다.
